# Kiểm tra ensemble 3 chuyên gia
Add Input → *Notebook Output* của 3 notebook chuyên gia, rồi **Run All** (không cần GPU).

In [ ]:
# ==============================================================================
# KIỂM TRA ENSEMBLE: 3 chuyên gia có thật sự khác nhau không, và đã học đủ 10 lớp chưa
# Add Input: Output của 3 notebook "bdd100k-rtdetr-expert-1/2/3". Chạy được trên CPU.
# ==============================================================================
import glob, json, os, itertools, torch
pths = sorted(glob.glob("/kaggle/input/**/expert_*.pth", recursive=True))
mans = sorted(glob.glob("/kaggle/input/**/expert_*_manifest.json", recursive=True))
reps = sorted(glob.glob("/kaggle/input/**/expert_*_class_report.txt", recursive=True))
print(f"[*] Tìm thấy {len(pths)} file trọng số, {len(mans)} manifest")
assert len(pths) >= 2, "Cần Output của ít nhất 2 chuyên gia trong Input."
ok = True
M = [json.load(open(p)) for p in mans]
for m in M:
    print(f"    chuyên gia {m['expert_id']}: seed={m['seed']} epoch={m['epochs_done']:.2f} sha={m['weights_sha256'][:12]}")
if len({m["seed"] for m in M}) < len(M) or len({m["weights_sha256"] for m in M}) < len(M):
    print("[X] Có chuyên gia trùng seed hoặc trùng trọng số!"); ok = False
sd = {p: torch.load(p, map_location="cpu", weights_only=True) for p in pths}
for a, b in itertools.combinations(pths, 2):
    for part, key in (("đầu phân loại", lambda k: "class_embed" in k or "enc_score_head" in k), ("backbone", lambda k: "backbone" in k)):
        ks = [k for k in sd[a] if key(k) and sd[a][k].is_floating_point()]
        va = torch.cat([sd[a][k].flatten().float() for k in ks]); vb = torch.cat([sd[b][k].flatten().float() for k in ks])
        d = ((va - vb).norm() / va.norm()).item()
        print(f"    {os.path.basename(a)} vs {os.path.basename(b)} | {part}: lệch tương đối {d:.4f}")
        if d < 1e-6:
            print("[X] Hai chuyên gia GIỐNG HỆT nhau!"); ok = False
for r in reps:
    print("\n" + open(r, encoding="utf-8").read())
    if "CHƯA HỌC ĐƯỢC" in open(r, encoding="utf-8").read():
        ok = False
print("\n[V] ENSEMBLE HỢP LỆ: các chuyên gia khác nhau và đã học đủ các lớp." if ok else "\n[X] ENSEMBLE CHƯA ĐẠT, xem các dòng [X] / CHƯA HỌC ĐƯỢC ở trên.")
